## 01. 교차검증

모델을 한 번만 train/test로 나누면 평가 점수가 데이터 분할에 따라 달라질 수 있다.
특히 데이터가 적거나 class 비율이 한쪽으로 치우친 경우에는 한 번의 점수만 보고 모델을 선택하기 어렵다.

**배우는 이유**
- 모델 성능을 더 안정적으로 추정하기 위함.
- 하이퍼파라미터 튜닝에서 테스트 데이터를 반복 사용하지 않기 위함.
- 여러 모델을 공정하게 비교하기 위함.

**어디서 사용하는가?**
- GridSearchCV, RandomizedSearchCV 같은 튜닝 도구 내부.
- 모델 후보를 비교할 때.
- 데이터가 많지 않아 검증셋을 따로 크게 떼기 어려울 때.

**핵심 용어**
- fold: 교차검증에서 데이터를 나눈 하나의 묶음.
- K-Fold: 데이터를 K개 fold로 나누고, 각 fold를 한 번씩 검증용으로 사용하는 방법.
- Stratified K-Fold: 분류 문제에서 class 비율을 유지하며 fold를 나누는 방법.
- validation score: 학습 데이터 내부 검증에서 얻은 점수.
- test score: 최종 평가 데이터에서 마지막으로 확인하는 점수.

## 02. 실습 환경 준비

Iris 분류 데이터와 Diabetes 회귀 데이터를 사용한다.
교차검증 함수, Pipeline, 스케일링, 분류/회귀 모델을 함께 사용해 검증 흐름을 확인한다.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_iris, load_diabetes

from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, cross_val_score, cross_validate

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression, Ridge

from sklearn.metrics import accuracy_score, f1_score

## 03. Iris 데이터 로드

Iris 데이터는 꽃받침/꽃잎 길이와 너비를 이용해 3개 품종 중 하나를 예측하는 분류 데이터셋이다.
교차검증에서는 class 비율이 fold마다 유지되는지 확인하기 좋다.

In [ ]:
iris = load_iris(as_frame=True)

iris_X = iris.data
iris_y = iris.target

print('feature shape:', iris_X.shape)
print('target shape:', iris_y.shape)
print('target names:', iris.target_names)
display(iris_X.head())
display(iris_y.value_counts().sort_index())


## 04. 학습/평가 데이터 분리

교차검증은 학습 데이터 내부에서 수행하고, 테스트 데이터는 마지막 확인용으로 남겨둔다.
`stratify=iris_y`를 사용하면 train/test에도 class 비율이 유지된다.

In [ ]:
iris_X_train, iris_X_test, iris_y_train, iris_y_test = train_test_split(
    iris_X,
    iris_y,
    test_size=0.2,
    random_state=42,
    stratify=iris_y
)

print('train:', iris_X_train.shape, iris_y_train.shape)
print('test:', iris_X_test.shape, iris_y_test.shape)
print('train class count')
display(iris_y_train.value_counts().sort_index())
print('test class count')
display(iris_y_test.value_counts().sort_index())


## 05. KFold와 StratifiedKFold 분할 비교

`KFold`는 단순히 데이터를 K개 묶음으로 나누고, `StratifiedKFold`는 class 비율을 유지하면서 나눈다.
분류 문제에서는 보통 `StratifiedKFold`를 우선 사용한다.

## 06. 단일 train/test 평가와 교차검증 비교

먼저 단일 train/test split으로 모델을 평가한 뒤, 같은 모델을 교차검증으로 평가한다.
두 결과를 비교하면 한 번의 점수보다 여러 fold 평균을 보는 이유가 드러남.

## 07. Pipeline과 cross_validate

`cross_validate()`는 여러 평가 지표와 학습/검증 시간을 함께 반환한다.
스케일링이 필요한 모델은 `Pipeline`으로 전처리와 모델을 묶어 교차검증 안에서 안전하게 처리한다.

## 08. 교차검증 후 최종 test 평가

교차검증으로 모델 후보의 성능을 추정한 뒤, 선택한 모델을 전체 train 데이터로 다시 학습하고 test 데이터로 마지막 평가한다.

## 09. 회귀 모델의 교차검증

교차검증은 분류뿐 아니라 회귀에도 사용한다.
회귀에서는 `R2`, `RMSE`, `MAE` 같은 지표를 사용하며, scikit-learn의 오차 지표는 큰 값이 좋다는 규칙 때문에 음수로 반환되는 경우가 있다.

## 10. 정리

- 교차검증은 학습 데이터를 여러 fold로 나누어 모델 성능을 더 안정적으로 추정하는 방법임.
- 분류 문제에서는 class 비율을 유지하는 StratifiedKFold를 자주 사용함.
- `cross_val_score()`는 하나의 지표를 간단히 확인할 때 사용함.
- `cross_validate()`는 여러 지표와 시간 정보를 함께 확인할 때 사용함.
- 테스트 데이터는 모델 선택과 튜닝에 반복 사용하지 않고 마지막 확인용으로 남겨야 함.